In [9]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

In [2]:
BASE_URL = "https://www.property24.co.ke/"


In [3]:
response = requests.get(BASE_URL)

In [6]:
soup = BeautifulSoup(response.text,'html.parser')


In [8]:
soup.select(".col-xs-6")

[<div class="col-xs-6">
 <ul class="col-xs-4">
 <li>
 <a href="/property-for-sale-in-nairobi-c1890" title="Property and houses for sale in Nairobi">Nairobi</a></li>
 <li>
 <a href="/property-for-sale-in-thika-c1850" title="Property and houses for sale in Thika">Thika</a></li>
 <li>
 <a href="/property-for-sale-in-kisumu-c1862" title="Property and houses for sale in Kisumu">Kisumu</a></li>
 <li>
 <a href="/property-for-sale-in-eldoret-c1919" title="Property and houses for sale in Eldoret">Eldoret</a></li>
 <li>
 <a href="/property-for-sale-in-malindi-c1853" title="Property and houses for sale in Malindi">Malindi</a></li>
 <li>
 <a href="/property-for-sale-in-kikuyu-c1847" title="Property and houses for sale in Kikuyu">Kikuyu</a></li>
 <li>
 <a href="/property-for-sale-in-kajiado-c1836" title="Property and houses for sale in Kajiado">Kajiado</a></li>
 <li>
 <a href="/property-for-sale-in-kwale-c1867" title="Property and houses for sale in Kwale">Kwale</a></li>
 <li>
 <a href="/property-f

In [11]:
location_elements = soup.select(".col-xs-6 ul li a")

locations = [
    {
        "name": link.get_text(strip=True),
        "url": f"{base_url}{link.get('href')}"
    }
    for link in location_elements
]

In [15]:
example_singel_page = locations[0]['url']

In [16]:
single_page_response = requests.get(example_singel_page)

In [20]:
single_page_soup = BeautifulSoup(single_page_response.text,"html.parser")

In [21]:
single_page_soup.select(".sc_searchResultsPagerTop")

[<div class="sc_searchResultsPagerTop">
 <div class="js_pager sc_panel sc_pager">
 <div class="btn-group pull-left">
 <button class="btn btn-default dropdown-toggle" data-testid="pager_top_sort_dropdown" data-toggle="dropdown" type="button">
 <span class="fontWeightBold">Order by:</span>  Default <i class="fa fa-caret-down"></i>
 </button>
 <ul class="dropdown-menu">
 <li>
 <a class="SortyByMenuItems" data-testid="pager_top_sort_dropdown_quality" data-value="Quality" href="/property-for-sale-in-nairobi-c1890?SortOrder=Quality" title="Default">Default</a>
 </li>
 <li>
 <a class="SortyByMenuItems" data-testid="pager_top_sort_dropdown_priceascending" data-value="PriceAscending" href="/property-for-sale-in-nairobi-c1890?SortOrder=PriceAscending" title="Price - low to high">Price - low to high</a>
 </li>
 <li>
 <a class="SortyByMenuItems" data-testid="pager_top_sort_dropdown_pricedescending" data-value="PriceDescending" href="/property-for-sale-in-nairobi-c1890?SortOrder=PriceDescending" ti

In [22]:
import math

# 1. Grab the <b> tags inside the page text container
# There are two <b> tags: the first is "1 - 20", the second is "126,807"
pager_tags = single_page_soup.select(".sc_searchResultsPagerTop .sc_pageText b")

if len(pager_tags) >= 2:
    # 2. Extract the raw text from the second <b> tag -> "126,807"
    raw_listings_count = pager_tags[1].get_text(strip=True)
    
    # 3. Clean the string! Casting "126,807" directly to an int will crash Python
    total_listings = int(raw_listings_count.replace(",", ""))
    
    # 4. Calculate total pages (20 items per page, rounding up)
    listings_per_page = 20
    total_pages = math.ceil(total_listings / listings_per_page)
    
    print(f"Raw String: '{raw_listings_count}'")
    print(f"Total Listings (int): {total_listings}")
    print(f"Total Pages to Scrape: {total_pages}")
else:
    print("Could not find the pager elements. Check if the page layout changed.")

Raw String: '126,807'
Total Listings (int): 126807
Total Pages to Scrape: 6341


In [39]:
listings = single_page_soup.find_all('div', class_='js_listingTile')

scraped_data = []

for listing in listings:
    # 1. Title / Property Type (e.g., "4 Bedroom Townhouse")
    title_elem = listing.find('span', class_='p24_propertyTitle')
    title = title_elem.text.strip() if title_elem else "N/A"
    
    # 2. Price (Extracting the clean numeric value from the 'content' attribute)
    price_elem = listing.find('span', class_='p24_price')
    price = price_elem['content'] if price_elem and 'content' in price_elem.attrs else "N/A"
    
    # 3. Location & Address
    location_elem = listing.find('span', class_='p24_location')
    location = location_elem.text.strip() if location_elem else "N/A"
    
    address_elem = listing.find('span', class_='p24_address')
    address = address_elem.text.strip() if address_elem else "N/A"
    
    # 4. Features (Beds, Baths, Parking, Size)
    # We look for the spans with specific titles, then grab the text from the inner <span>
    
    bed_elem = listing.find('span', title='Bedrooms')
    bedrooms = bed_elem.find('span').text.strip() if bed_elem else "N/A"
    
    bath_elem = listing.find('span', title='Bathrooms')
    bathrooms = bath_elem.find('span').text.strip() if bath_elem else "N/A"
    
    park_elem = listing.find('span', title='Parking Spaces')
    parking = park_elem.find('span').text.strip() if park_elem else "N/A"
    
    size_elem = listing.find('span', title='Floor Size')
    floor_size = size_elem.find('span').text.strip() if size_elem else "N/A"
    
    # Append the extracted features as a dictionary to our list
    scraped_data.append({
        'Title': title,
        'Price (KSh)': price,
        'Location': location,
        'Address': address,
        'Bedrooms': bedrooms,
        'Bathrooms': bathrooms,
        'Parking': parking,
        'Floor Size': floor_size
    })

# Print the first extracted item to verify
print(scraped_data[5])

{'Title': '4 Bedroom Townhouse', 'Price (KSh)': '169000000', 'Location': 'Lavington', 'Address': 'Mugumo Rd Kileleshwa, Lavington, Nairobi', 'Bedrooms': '4', 'Bathrooms': '5', 'Parking': '3', 'Floor Size': '800 m²'}
